# Sub-sampling configuration sensitivityHow performance and runtime respond to the pairs-per-anchor `s` and the pair mini-batch size `b`.## What changed in this notebookThe model, loss, sampler, metrics and data generation used to be defined**inline in this notebook**, and the same block was copy-pasted into five othernotebooks. That is why two defects survived so long: fixing one copy left theothers untouched, and the copies had already drifted apart.All of that now lives in the `rnn_agt` package. This notebook only sets up anexperiment and reports it.Two fixes are inherited automatically:1. **Censoring now reaches the outcome.** The old `prepare_subjects_for_nn`   passed `subj['log_gaps']` — the *latent, uncensored* gap times — to the   model, while `delta` said some records were censored. Padding past the   censoring point was passed through as real data too.2. **The WRS normalization `1/(K_i* K_l*)` is applied.** The old loss was a   plain Gehan rank loss. The subject-level weight is the mechanism that   handles induced dependent censoring, so without it the estimating function   is biased toward subjects with many events.`simulation/defect_impact.ipynb` measures how much both defects changed the numbers.**Also relevant here:** the old sampler called `np.delete(all_flat, flat)`inside a Python loop over every uncensored record, allocating a freshN-element array per anchor. The vectorised sampler makes this sweep tractable.

In [ ]:
import os, syssys.path.insert(0, os.path.abspath(".."))   # repository root, so `rnn_agt` importsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport rnn_agtfrom rnn_agt import data as Dfrom rnn_agt.seeds import make_seedsfrom rnn_agt.train import TrainConfig, train_model, predictfrom rnn_agt.metrics import evaluateprint("rnn_agt", rnn_agt.__version__)

In [ ]:
import timefrom rnn_agt.diagnostics import check_subsampling_unbiasednessseeds = make_seeds(2026)rng = seeds.data()tau = D.calibrate_tau(1000, D.f_interaction, "normal", rng, 0.50,                      D.DEPENDENCE_SPECS["ar1"])train_subjects = D.make_dataset(1000, "interaction", "normal", rng,                                dependence="ar1", tau=tau)test_subjects  = D.make_dataset(2000, "interaction", "normal", rng,                                dependence="ar1", tau=tau)print(f"{len(train_subjects)} train subjects, tau={tau:.1f}")

### Unbiasedness holds regardless of `s`Theorem A.2 says the subsampled objective is unbiased for the full one at any `s`; `s` controls variance, not bias. Confirm that before interpreting the sweep, since a biased estimator would make the whole comparison meaningless.

In [ ]:
for s in (2, 5, 10, 30):    chk = check_subsampling_unbiasedness(train_subjects[:120], 3, n_draws=300, s=s)    print(f"s={s:3d}  exact={chk['exact']:.3f}  MC mean={chk['mc_mean']:.3f}  "          f"ratio={chk['ratio']:.4f}  z={chk['z']:+.2f}")

In [ ]:
S_GRID = [2, 5, 10, 30, 50]B_GRID = [32, 64, 128]rows = []for s in S_GRID:    for b in B_GRID:        cfg = TrainConfig(model="rnn_agt", epochs=10, pair_sample_s=s,                          pair_batch_b=b, hidden_dim=64, gru_layers=2, lr=3e-4)        t0 = time.time()        res = train_model(train_subjects, test_subjects, 3, cfg, make_seeds(11))        rows.append({            "s": s, "b": b,            "test C": res.metrics["test_cindex"],            "test AMSE": res.metrics["test_amse"],            "seconds": time.time() - t0,        })        print(f"s={s:3d} b={b:4d}  C={rows[-1]['test C']:.3f}  "              f"AMSE={rows[-1]['test AMSE']:.2f}  {rows[-1]['seconds']:.1f}s",              flush=True)subsample = pd.DataFrame(rows)subsample.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))for b in B_GRID:    sub = subsample[subsample.b == b]    axes[0].plot(sub.s, sub["test C"], marker="o", label=f"b={b}")    axes[1].plot(sub.s, sub["seconds"], marker="o", label=f"b={b}")axes[0].set_xlabel("pairs per anchor, s"); axes[0].set_ylabel("test IPCW C-index")axes[1].set_xlabel("pairs per anchor, s"); axes[1].set_ylabel("runtime (s)")for ax in axes:    ax.legend(); ax.grid(alpha=.3)fig.suptitle("Sub-sampling configuration: accuracy against cost")fig.tight_layout()fig.savefig("subsampling_sensitivity.png", dpi=150)plt.show()print("Look for the smallest s where the C-index curve flattens; beyond it you "      "are paying runtime for variance reduction that no longer moves the "      "estimate.")